<p align="center">
  <img src="./images/HealthOmics_logo.png" alt="Logo" style="width: 100px;">
</p>

<h1 align="center">AWS HealthOmics Tutorial</h1>

## Overview

<img src="./images/HealthOmics-Overview.png" alt="HealthOmics Overview" width="75%" height="500px">

### Introduction to AWS HealthOmics

AWS HealthOmics is a comprehensive suite of services designed to facilitate the storage, analysis, and sharing of genomic, transcriptomic, and other omics data. It provides scalable and secure infrastructure to support bioinformatics workflows, enabling researchers and clinicians to derive meaningful insights from complex biological data.

#### Components of AWS HealthOmics

- **AWS Omics Storage**: Secure and scalable storage solutions for omics data, ensuring data integrity and compliance with regulatory requirements.
- **AWS Omics Analytics**: Powerful tools and services for the analysis of omics data, including machine learning and AI capabilities to enhance data interpretation.
- **AWS Omics Workflows**: Streamlined workflow management for running complex bioinformatics pipelines efficiently and reproducibly.

#### Why Use AWS HealthOmics for Bioinformatics Pipelines?

- **Scalability**: AWS HealthOmics can handle large-scale datasets, making it ideal for high-throughput sequencing projects.
- **Security**: Robust security measures ensure that sensitive biological data is protected.
- **Cost-Effectiveness**: Pay-as-you-go pricing models help manage costs effectively.
- **Integration**: Seamless integration with other AWS services, such as AWS Lambda, AWS Batch, and Amazon S3, to streamline workflow management.
- **Reproducibility**: Ensures reproducibility of bioinformatics analyses through containerization and version control.

### Key Steps for Creating and Running a Private Workflow

<img src="./images/HealthOmics-Workflow.png" alt="HealthOmics Overview" width="75%" height="500px">

1. **Set Up AWS Environment**:
   - Create an AWS account and configure necessary IAM roles and permissions.
   - Set up AWS S3 buckets for data storage.

2. **Install Required Tools**:
   - Install Nextflow and other dependencies like Docker, Singularity, or Conda.

3. **Prepare Workflow Scripts**:
   - Develop or obtain Nextflow scripts for the desired bioinformatics pipeline.

4. **Configure Workflow**:
   - Customize configuration files to specify computational resources and environment settings.
   - Ensure all containers are stored in an Amazon ECR (Elastic Container Registry) instance.

5. **Run the Workflow**:
   - Execute the workflow using Nextflow, specifying the input data and output directories.

6. **Monitor and Manage Jobs**:
   - Use AWS Batch or other job management tools to monitor workflow execution and manage computational resources.

7. **Analyze Results**:
   - Retrieve and analyze the results from the specified output directory.


### Tutorial Walkthrough: Methylseq Pipeline

This tutorial will guide you through the process of setting up and running the nf-core/methylseq pipeline on AWS HealthOmics. The nf-core/methylseq pipeline is a bioinformatics analysis pipeline used for Methylation (Bisulfite) sequencing data. It pre-processes raw data from FastQ inputs, aligns the reads, and performs extensive quality control on the results.

The pipeline supports the Bismark Workflow:
- **Bismark Workflow**: Utilizes Bismark for alignment and methylation extraction.

By the end of this tutorial, you will have a comprehensive understanding of how to implement and run the methylseq pipeline on AWS HealthOmics, leveraging its robust infrastructure to achieve reproducible and scalable bioinformatics analyses.


## IAM Roles and Permissions

Setting up the correct IAM (Identity and Access Management) roles and permissions is crucial for ensuring secure and efficient access to AWS HealthOmics services. Below are the key steps and permissions required:

### 1. Create an IAM Role for AWS HealthOmics

1. **Sign in to the AWS Management Console** and open the IAM console at [https://console.aws.amazon.com/iam/](https://console.aws.amazon.com/iam/).
2. **In the navigation pane, choose Roles**, and then choose **Create role**.
3. **Select AWS service** as the type of trusted entity, and then choose **EC2** (or the relevant service you will be using).
4. **Attach the necessary policies**. You can start with the following managed policies:
   - `AmazonS3FullAccess`: Grants full access to Amazon S3 buckets.
   - `AmazonEC2FullAccess`: Grants full access to Amazon EC2 instances.
   - `AWSBatchFullAccess`: Grants full access to AWS Batch, if you are using it for job management.
   - `AWSLambdaFullAccess`: Grants full access to AWS Lambda, if you are using it for serverless workflows.
   - `AmazonECRFullAccess`: Grants full access to Amazon Elastic Container Registry.
   - `AWSHealthOmicsFullAccess`: Grants full access to AWS HealthOmics services.
5. **Review and create the role**. Provide a role name (e.g., `HealthOmicsRole`) and review the permissions before creating the role.

### 2. Create an IAM Policy for Custom Permissions

If you need more granular control over permissions, you can create a custom IAM policy:

1. **In the IAM console, choose Policies**, and then choose **Create policy**.
2. **Use the JSON editor** to define the custom policy. Below is an example policy that grants specific permissions:

   ```json
   {
     "Version": "2012-10-17",
     "Statement": [
       {
         "Effect": "Allow",
         "Action": [
           "s3:ListBucket",
           "s3:GetObject",
           "s3:PutObject",
           "ec2:DescribeInstances",
           "ec2:StartInstances",
           "ec2:StopInstances",
           "batch:SubmitJob",
           "batch:DescribeJobs",
           "lambda:InvokeFunction",
           "ecr:GetDownloadUrlForLayer",
           "ecr:BatchGetImage",
           "omics:*"
         ],
         "Resource": "*"
       }
     ]
   }


## Storage Setup

Proper storage setup is essential for managing and accessing omics data efficiently in AWS HealthOmics. There are two primary storage options available: Amazon S3 and AWS Omics Storage. Below are the key steps and considerations for setting up each storage option:

### Option 1: Amazon S3

Amazon S3 (Simple Storage Service) is a scalable and secure storage solution for storing omics data. Follow these steps to create an S3 bucket:

#### 1. Create an S3 Bucket

1. **Sign in to the AWS Management Console** and open the S3 console at [https://console.aws.amazon.com/s3/](https://console.aws.amazon.com/s3/).
2. **Choose Create bucket**.
3. **Configure the bucket settings**:
   - **Bucket name**: Provide a unique name for your bucket (e.g., `healthomics-data-bucket`).
   - **Region**: Choose the AWS Region where you want to create the bucket.
4. **Configure options**: You can configure additional settings such as versioning, logging, and encryption based on your requirements.
5. **Review and create the bucket**.

#### 2. Set Up Bucket Policies and Permissions

Ensure that your S3 bucket has the appropriate policies and permissions to allow access from your AWS HealthOmics services:

1. **In the S3 console, choose the bucket you created**.
2. **Choose the Permissions tab**, and then choose **Bucket policy**.
3. **Add a bucket policy** to grant access to your IAM role. Below is an example policy:

   ```json
   {
     "Version": "2012-10-17",
     "Statement": [
       {
         "Effect": "Allow",
         "Principal": {
           "AWS": "arn:aws:iam::YOUR_ACCOUNT_ID:role/HealthOmicsRole"
         },
         "Action": [
           "s3:GetObject",
           "s3:PutObject",
           "s3:ListBucket"
         ],
         "Resource": [
           "arn:aws:s3:::healthomics-data-bucket",
           "arn:aws:s3:::healthomics-data-bucket/*"
         ]
       }
     ]
   }
### Option 2: AWS Omics Storage

<img src="./images/HealthOmics-Storage.png" alt="HealthOmics Overview" width="75%" height="500px">


AWS Omics Storage is a specialized storage solution designed specifically for omics data, providing optimized performance and features for bioinformatics workflows. Follow these steps to set up AWS Omics Storage:

#### 1. Create an Omics Store

1. **Sign in to the AWS Management Console** and open the Omics console at [https://console.aws.amazon.com/omics/](https://console.aws.amazon.com/omics/).
2. **Choose Create store**.
3. **Configure the store settings**:
   - **Store name**: Provide a unique name for your store (e.g., `healthomics-store`).
   - **Region**: Choose the AWS Region where you want to create the store.
   - **Store type**: Select the appropriate store type based on your data (e.g., variant store, annotation store).
4. **Review and create the store**.

#### 2. Set Up Store Policies and Permissions

Ensure that your Omics store has the appropriate policies and permissions to allow access from your AWS HealthOmics services:

1. **In the Omics console, choose the store you created**.
2. **Choose the Permissions tab**, and then choose **Store policy**.
3. **Add a store policy** to grant access to your IAM role. Below is an example policy:

   ```json
   {
     "Version": "2012-10-17",
     "Statement": [
       {
         "Effect": "Allow",
         "Principal": {
           "AWS": "arn:aws:iam::YOUR_ACCOUNT_ID:role/HealthOmicsRole"
         },
         "Action": [
           "omics:GetStore",
           "omics:PutStore",
           "omics:ListStores"
         ],
         "Resource": [
           "arn:aws:omics:REGION:YOUR_ACCOUNT_ID:store/healthomics-store"
         ]
       }
     ]
   }

## Workflow Setup

Setting up workflows in AWS HealthOmics is crucial for automating and managing bioinformatics tasks efficiently. AWS HealthOmics supports several workflow languages, including Nextflow, WDL (Workflow Description Language), and CWL (Common Workflow Language). Below are the key steps and considerations for setting up workflows:

### 1. Supported Workflow Languages

AWS HealthOmics supports the following workflow languages:

- **Nextflow**: A workflow management system that simplifies writing and deploying complex computational pipelines.
- **WDL (Workflow Description Language)**: A human-readable and writable language designed for describing data processing workflows.
- **CWL (Common Workflow Language)**: A standard for describing analysis workflows and tools in a way that makes them portable and scalable.

### 2. Preparing Your Workflow

Before uploading your workflow to AWS HealthOmics, ensure that it is properly prepared and zipped. Follow these generalized steps:

1. **Write Your Workflow Script**:
   - Create a script file containing your workflow (e.g., `main.nf` for Nextflow, `workflow.wdl` for WDL, `workflow.cwl` for CWL).

2. **Include Configuration and Input Files**:
   - Include any necessary configuration files (e.g., `nextflow.config` for Nextflow, JSON input files for WDL, YAML input files for CWL).
   - Create a `parameters.json` file to specify the parameters for your workflow.

3. **Organize Your Workflow Directory**:
   - Organize your workflow files into a directory structure. For example:
     ```
     my-workflow/
     ├── main.nf (or workflow.wdl, workflow.cwl)
     ├── nextflow.config (or inputs.json, inputs.yaml)
     ├── parameters.json
     └── other_files/
         ├── script1.sh
         └── script2.sh
     ```

4. **Zip the Workflow Directory**:
   - Compress the workflow directory into a ZIP file. For example:
     ```
     zip -r my-workflow.zip my-workflow/
     ```

### 3. Uploading Your Workflow to AWS HealthOmics

Once your workflow is zipped, you can upload it to AWS HealthOmics:

1. **Sign in to the AWS Management Console** and open the HealthOmics console at [https://console.aws.amazon.com/omics/](https://console.aws.amazon.com/omics/).
2. **Choose Workflows** from the navigation pane.
3. **Choose Create workflow**.
4. **Upload the ZIP file** containing your workflow.
5. **Configure the workflow settings**:
   - **Name**: Provide a unique name for your workflow.
   - **Description**: Optionally, provide a description for your workflow.
   - **Language**: Select the workflow language (e.g., Nextflow, WDL, CWL).
6. **Review and create the workflow**.

### 4. Running Your Workflow

After uploading and configuring your workflow, you can run it:

1. **Choose the workflow** you created from the list of workflows.
2. **Choose Run workflow**.
3. **Configure the run settings**:
   - **Input Data**: Specify the input data files and parameters.
   - **Parameters File**: Upload the `parameters.json` file.
   - **Output Location**: Specify the S3 bucket for storing output data.
4. **Start the workflow run**.

## ECR Setup

Amazon Elastic Container Registry (ECR) is a fully managed Docker container registry that makes it easy to store, manage, and deploy Docker container images. For AWS HealthOmics workflows, you may need to pull public containers and store them in ECR to ensure efficient and secure access. Below are the key steps and considerations for setting up ECR and connecting it to AWS HealthOmics:

### 1. Create an ECR Repository

1. **Sign in to the AWS Management Console** and open the ECR console at [https://console.aws.amazon.com/ecr/](https://console.aws.amazon.com/ecr/).
2. **Choose Create repository**.
3. **Configure the repository settings**:
   - **Repository name**: Provide a unique name for your repository (e.g., `healthomics-containers`).
   - **Visibility settings**: Choose `Private` to restrict access to your AWS account.
4. **Review and create the repository**.

### 2. Pull Public Containers and Push to ECR

To pull public containers and store them in your ECR repository, follow these steps:

1. **Authenticate Docker to Your ECR Registry**:
   - Use the AWS CLI to authenticate Docker to your ECR registry. Replace `REGION` with your AWS region.
     ```sh
     aws ecr get-login-password --region REGION | docker login --username AWS --password-stdin ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com
     ```

2. **Pull the Public Container Image**:
   - Use Docker to pull the public container image. For example, to pull a public image from Docker Hub:
     ```sh
     docker pull public-image:tag
     ```

3. **Tag the Image for Your ECR Repository**:
   - Tag the pulled image with your ECR repository URI. Replace `ACCOUNT_ID`, `REGION`, `REPOSITORY_NAME`, and `IMAGE_TAG` with your specific details.
     ```sh
     docker tag public-image:tag ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com/REPOSITORY_NAME:IMAGE_TAG
     ```

4. **Push the Image to Your ECR Repository**:
   - Push the tagged image to your ECR repository.
     ```sh
     docker push ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com/REPOSITORY_NAME:IMAGE_TAG
     ```

### 3. Set Up Permissions for ECR

Ensure that your AWS HealthOmics services have the appropriate permissions to access your ECR repository:

1. **Create an IAM Role**:
   - Create an IAM role that grants permissions to access your ECR repository. Attach the following policy to the role:
     ```json
     {
       "Version": "2012-10-17",
       "Statement": [
         {
           "Effect": "Allow",
           "Action": [
             "ecr:GetDownloadUrlForLayer",
             "ecr:BatchGetImage",
             "ecr:BatchCheckLayerAvailability"
           ],
           "Resource": "arn:aws:ecr:REGION:ACCOUNT_ID:repository/REPOSITORY_NAME"
         }
       ]
     }
     ```

2. **Attach the IAM Role to Your HealthOmics Services**:
   - Ensure that the IAM role is attached to the services or compute environments used by your AWS HealthOmics workflows.

### 4. Connect ECR to AWS HealthOmics Workflows

To use the container images stored in ECR within your AWS HealthOmics workflows, specify the ECR image URI in your workflow definition:

1. **Update Your Workflow Script**:
   - In your workflow script (Nextflow, WDL, or CWL), specify the ECR image URI for the tasks that require the container image. For example:
     ```json
     {
       "container": "ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com/REPOSITORY_NAME:IMAGE_TAG"
     }
     ```

2. **Run Your Workflow**:
   - Ensure that your workflow is configured to use the ECR image and run it as usual.

By following these steps, you will set up Amazon ECR to store public container images and connect them to your AWS HealthOmics workflows, ensuring efficient and secure access to the necessary container images.


## Running Your Workflow

After uploading and configuring your workflow and setting up ECR, you can run your workflow in AWS HealthOmics:

1. **Choose the workflow** you created from the list of workflows.
2. **Choose Run workflow**.
3. **Configure the run settings**:
   - **Input Data**: Specify the input data files and parameters.
   - **Parameters File**: Upload the `parameters.json` file.
   - **Output Location**: Specify the S3 bucket for storing output data.
4. **Start the workflow run**.

By following these steps, you will set up and manage efficient workflows for AWS HealthOmics, ensuring automated and streamlined bioinformatics processes.


## Runs and Run Groups

In AWS HealthOmics, managing and organizing workflow executions is essential for efficient bioinformatics analysis. This is achieved through the concepts of runs and run groups. Understanding these concepts will help you better manage your workflow executions and analyze results more effectively.

### 1. Runs

A run is a single execution of a workflow. Each run processes a specific set of input data and parameters, producing output data that can be analyzed and stored. Key aspects of runs include:

- **Run ID**: A unique identifier for each run, which helps in tracking and managing individual executions.
- **Input Data**: The data files and parameters provided for the run.
- **Output Data**: The results generated by the run, typically stored in an S3 bucket.
- **Status**: The current state of the run (e.g., running, succeeded, failed).
- **Logs**: Detailed logs of the run's execution, useful for debugging and analysis.

### 2. Run Groups

Run groups allow you to organize multiple runs into logical groups. This is particularly useful when you have multiple runs that are related or when you want to analyze and compare results across different runs. Key aspects of run groups include:

- **Group ID**: A unique identifier for each run group.
- **Group Name**: A descriptive name for the run group, helping in identifying the purpose or context of the group.
- **Runs**: A collection of runs that belong to the group.
- **Metadata**: Additional information about the run group, such as descriptions or tags.


### 3. Creating and Managing Run Groups

To create and manage run groups in AWS HealthOmics, follow these steps:

1. **Create a Run Group**:
   - In the AWS HealthOmics console, navigate to the Run Groups section.
   - Choose **Create run group**.
   - Provide a **Group Name** and any additional metadata.
   - Save the run group.

2. **Add Runs to a Run Group**:
   - After creating a run, you can add it to a run group.
   - Navigate to the run details page.
   - Choose **Add to run group** and select the appropriate run group.

3. **Manage Run Groups**:
   - View and manage run groups from the Run Groups section in the AWS HealthOmics console.
   - Analyze and compare results across different runs within a group.
   - Use metadata and tags to organize and filter run groups.

### 4. Benefits of Using Run Groups

Using run groups offers several benefits:

- **Organization**: Group related runs together for better organization and management.
- **Comparison**: Easily compare results across different runs within a group.
- **Analysis**: Perform collective analysis on runs within a group to identify patterns and insights.
- **Efficiency**: Streamline the management of multiple runs, reducing complexity and improving efficiency.

By understanding and utilizing runs and run groups in AWS HealthOmics, you can effectively manage your workflow executions and gain deeper insights into your bioinformatics analyses.


## AWS HealthOmics Analytics, Variant Stores, and Annotation Stores

AWS HealthOmics provides powerful tools for analyzing genomic data, managing variant stores, and utilizing annotation stores. These features enable researchers to efficiently process, store, and analyze large-scale genomic data, facilitating advanced bioinformatics research and clinical applications.


<img src="./images/HealthOmics-Analytics.png" alt="HealthOmics Overview" width="75%" height="500px">

### 1. AWS HealthOmics Analytics

AWS HealthOmics Analytics offers a suite of tools and services designed to analyze genomic data at scale. Key features include:

- **Scalability**: Leverage AWS's scalable infrastructure to process large volumes of genomic data.
- **Integration**: Seamlessly integrate with other AWS services, such as Amazon S3, Amazon EC2, and AWS Lambda.
- **Data Security**: Ensure data security and compliance with industry standards and regulations.
- **Cost Efficiency**: Optimize costs with pay-as-you-go pricing and resource management.

### 2. Variant Stores

Variant stores in AWS HealthOmics are specialized databases designed to store and manage genomic variants. These stores enable efficient querying and analysis of variant data. Key aspects of variant stores include:

- **Storage**: Store large volumes of variant data in a structured format.
- **Querying**: Perform efficient queries on variant data using SQL-like syntax.
- **Integration**: Integrate with other AWS HealthOmics services for seamless data processing and analysis.
- **Scalability**: Scale storage and querying capabilities to handle large datasets.

#### Creating a Variant Store

To create a variant store in AWS HealthOmics, follow these steps:

1. **Sign in to the AWS Management Console** and open the HealthOmics console at [https://console.aws.amazon.com/omics/](https://console.aws.amazon.com/omics/).
2. **Choose Variant Stores** from the navigation pane.
3. **Choose Create variant store**.
4. **Configure the variant store settings**:
   - **Name**: Provide a unique name for your variant store.
   - **Description**: Optionally, provide a description for your variant store.
   - **Storage Location**: Specify the S3 bucket for storing variant data.
5. **Review and create the variant store**.

#### Managing Variant Data

Once the variant store is created, you can manage variant data by:

- **Uploading Variant Data**: Use the AWS CLI or SDKs to upload variant data files to the specified S3 bucket.
- **Querying Variant Data**: Use the AWS HealthOmics console or APIs to perform queries on the variant data.
- **Analyzing Variant Data**: Integrate with AWS HealthOmics Analytics to perform advanced analyses on the variant data.

### 3. Annotation Stores

Annotation stores in AWS HealthOmics are specialized databases designed to store and manage genomic annotations. These stores enable efficient querying and analysis of annotation data. Key aspects of annotation stores include:

- **Storage**: Store large volumes of annotation data in a structured format.
- **Querying**: Perform efficient queries on annotation data using SQL-like syntax.
- **Integration**: Integrate with other AWS HealthOmics services for seamless data processing and analysis.
- **Scalability**: Scale storage and querying capabilities to handle large datasets.

#### Creating an Annotation Store

To create an annotation store in AWS HealthOmics, follow these steps:

1. **Sign in to the AWS Management Console** and open the HealthOmics console at [https://console.aws.amazon.com/omics/](https://console.aws.amazon.com/omics/).
2. **Choose Annotation Stores** from the navigation pane.
3. **Choose Create annotation store**.
4. **Configure the annotation store settings**:
   - **Name**: Provide a unique name for your annotation store.
   - **Description**: Optionally, provide a description for your annotation store.
   - **Storage Location**: Specify the S3 bucket for storing annotation data.
5. **Review and create the annotation store**.

#### Managing Annotation Data

Once the annotation store is created, you can manage annotation data by:

- **Uploading Annotation Data**: Use the AWS CLI or SDKs to upload annotation data files to the specified S3 bucket.
- **Querying Annotation Data**: Use the AWS HealthOmics console or APIs to perform queries on the annotation data.
- **Analyzing Annotation Data**: Integrate with AWS HealthOmics Analytics to perform advanced analyses on the annotation data.

### 4. Benefits of Using Variant and Annotation Stores

Using variant and annotation stores in AWS HealthOmics offers several benefits:

- **Efficiency**: Efficiently store and manage large volumes of genomic data.
- **Scalability**: Scale storage and querying capabilities to handle large datasets.
- **Integration**: Seamlessly integrate with other AWS HealthOmics services for comprehensive data analysis.
- **Advanced Analytics**: Perform advanced analyses on variant and annotation data to gain deeper insights into genomic research.

By leveraging AWS HealthOmics Analytics, variant stores, and annotation stores, researchers can efficiently process, store, and analyze large-scale genomic data, facilitating advanced bioinformatics research and clinical applications.
